# TRRUST v2 TF-Target Interactions Preprocessing for QuVINE

This notebook helps you:

1. Download TRRUST v2 TF-target interactions (human and/or mouse)
2. Parse signed regulation (Activation, Repression, Unknown)
3. Convert to a clean undirected NetworkX graph
4. Extract repeated **connected induced subgraphs** of approximately **5k–20k nodes**
5. Use a degree-aware sampling heuristic to better preserve the original degree distribution
6. Save sampled subgraphs as edge lists and metadata for downstream QuVINE processing

## About TRRUST v2

TRRUST (Transcriptional Regulatory Relationships Unraveled by Sentence-based Text mining) is a curated database of human and mouse transcriptional regulatory networks. Each interaction includes:
- **TF**: Transcription factor
- **Target**: Target gene
- **Effect**: Regulation type (Activation, Repression, Unknown)
- **PMID**: PubMed ID reference

## Design choices

- **Data source**: TRRUST v2 from GitHub mirror
- **Species**: Human (default), Mouse (optional)
- **Effect filtering**: Option to drop "Unknown" effects
- **Sampling goal**: connected, induced, degree-aware, repeated multiple times (e.g. 5 or 10 replicas)
- **Output format**: CSV edgelists + JSON metadata

## 1. Environment and imports

In [ ]:
# Install missing packages if needed
# %pip install pandas networkx numpy matplotlib scipy tqdm requests

In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter, deque
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
# TRRUST v2 data sources
HUMAN_URL = "https://raw.githubusercontent.com/bioinfonerd/Transcription-Factor-Databases/master/Ttrust_v2/trrust_rawdata.human.tsv"
MOUSE_URL = "https://raw.githubusercontent.com/bioinfonerd/Transcription-Factor-Databases/master/Ttrust_v2/trrust_rawdata.mouse.tsv.gz"

# Choose species: "human" or "mouse"
SPECIES = "human"

# Drop "Unknown" regulation effects?
DROP_UNKNOWN = False  # Set to True to keep only Activation/Repression

# Output location
OUTPUT_DIR = Path(f"../data/processed_data/trrust_{SPECIES}_graph_samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sampling settings
TARGET_SIZES = [5000, 10000, 20000]
REPEATS = 5  # set to 10 if desired
BASE_SEED = 42

# Degree-aware sampling heuristic settings
RESTART_PROB = 0.15
FRONTIER_WIDTH = 256
DEGREE_WEIGHT_POWER = 0.5  # >0 prefers higher-degree anchors somewhat
LOCAL_BFS_EXPANSION = 8

## 3. Download and inspect TRRUST data

In [ ]:
# Download TRRUST data
if SPECIES.lower() == "human":
    url = HUMAN_URL
    compression = None
    print(f"Downloading TRRUST v2 human data from: {url}")
elif SPECIES.lower() == "mouse":
    url = MOUSE_URL
    compression = "gzip"
    print(f"Downloading TRRUST v2 mouse data from: {url}")
else:
    raise ValueError(f"Unknown species: {SPECIES}. Choose 'human' or 'mouse'.")

# Load data (no header in raw files)
trrust = pd.read_csv(
    url,
    sep="\t",
    compression=compression,
    header=None,
    names=["TF", "Target", "Effect", "PMID"]
)

print(f"\nDataset shape: {trrust.shape}")
print(f"Columns: {list(trrust.columns)}")
trrust.head(10)

## 4. Analyze regulation effects

In [ ]:
# Analyze effect types
print("Regulation effect distribution:")
print(trrust["Effect"].value_counts())
print(f"\nTotal interactions: {len(trrust)}")

# Show examples of each effect type
for effect in trrust["Effect"].unique():
    print(f"\nExample {effect} interactions:")
    print(trrust[trrust["Effect"] == effect][["TF", "Target", "Effect"]].head(3))

## 5. Filter data (optional)

In [ ]:
# Filter out "Unknown" effects if requested
if DROP_UNKNOWN:
    trrust_filtered = trrust[trrust["Effect"].str.lower() != "unknown"].copy()
    print(f"Original dataset: {trrust.shape[0]} interactions")
    print(f"After dropping 'Unknown': {trrust_filtered.shape[0]} interactions")
    print(f"Reduction: {100 * (1 - trrust_filtered.shape[0] / trrust.shape[0]):.1f}%")
else:
    trrust_filtered = trrust.copy()
    print(f"Using all {trrust_filtered.shape[0]} interactions (including 'Unknown' effects)")

print(f"\nFinal effect distribution:")
print(trrust_filtered["Effect"].value_counts())

## 6. Build NetworkX graph from TRRUST interactions

In [ ]:
def build_graph_from_trrust(df: pd.DataFrame, tf_col: str = "TF", 
                            target_col: str = "Target") -> nx.Graph:
    """
    Build an undirected NetworkX graph from TRRUST TF-target interactions.
    
    Args:
        df: DataFrame with TF-target interactions
        tf_col: Column name for transcription factors
        target_col: Column name for target genes
    
    Returns:
        Undirected simple graph (no self-loops, no multi-edges)
    """
    # Create directed graph first (TF -> target)
    # Note: We keep edge attributes (Effect, PMID) for potential future use
    G_directed = nx.from_pandas_edgelist(
        df,
        source=tf_col,
        target=target_col,
        edge_attr=True,
        create_using=nx.DiGraph()
    )
    
    # Convert to undirected and remove self-loops
    G = G_directed.to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    
    return G


def keep_largest_connected_component(G: nx.Graph) -> nx.Graph:
    """Extract the largest connected component."""
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    lcc_nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(lcc_nodes).copy()


# Build the graph
G_full = build_graph_from_trrust(trrust_filtered, tf_col="TF", target_col="Target")
print(f"Initial graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges")
print(f"Graph type: {type(G_full).__name__} (undirected={not G_full.is_directed()})")
print(f"Connected components: {nx.number_connected_components(G_full)}")

# Extract largest connected component
G_full = keep_largest_connected_component(G_full)
print(f"\nLargest connected component:")
print(f"  Nodes: {G_full.number_of_nodes()}")
print(f"  Edges: {G_full.number_of_edges()}")
print(f"  Connected: {nx.is_connected(G_full)}")
print(f"  Average degree: {2 * G_full.number_of_edges() / G_full.number_of_nodes():.2f}")

# Verify it's undirected
assert not G_full.is_directed(), "ERROR: Graph should be undirected!"
print("\n✓ Verified: Graph is undirected")

## 7. Graph analysis helpers

In [ ]:
def graph_degree_array(G: nx.Graph) -> np.ndarray:
    if G.number_of_nodes() == 0:
        return np.array([], dtype=float)
    return np.array([d for _, d in G.degree()], dtype=float)


def summarize_graph(G: nx.Graph) -> Dict[str, float]:
    deg = graph_degree_array(G)
    return {
        "num_nodes": int(G.number_of_nodes()),
        "num_edges": int(G.number_of_edges()),
        "avg_degree": float(deg.mean()) if len(deg) else 0.0,
        "median_degree": float(np.median(deg)) if len(deg) else 0.0,
        "max_degree": float(deg.max()) if len(deg) else 0.0,
        "num_connected_components": int(nx.number_connected_components(G)),
    }


def plot_degree_ccdf(graphs: Dict[str, nx.Graph], title: str = "Degree CCDF comparison"):
    plt.figure(figsize=(7, 5))
    for label, G in graphs.items():
        deg = graph_degree_array(G)
        deg = deg[deg > 0]
        if len(deg) == 0:
            continue
        xs = np.sort(np.unique(deg))
        ccdf = np.array([(deg >= x).mean() for x in xs])
        plt.step(xs, ccdf, where="post", label=label)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Degree")
    plt.ylabel("CCDF")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def degree_histogram_distance(G_ref: nx.Graph, G_sub: nx.Graph, bins: int = 30) -> float:
    deg_ref = graph_degree_array(G_ref)
    deg_sub = graph_degree_array(G_sub)
    if len(deg_ref) == 0 or len(deg_sub) == 0:
        return float("inf")
    upper = max(float(deg_ref.max()), float(deg_sub.max()), 1.0)
    bin_edges = np.linspace(0.0, upper, bins + 1)
    h_ref, _ = np.histogram(deg_ref, bins=bin_edges, density=True)
    h_sub, _ = np.histogram(deg_sub, bins=bin_edges, density=True)
    return float(np.abs(h_ref - h_sub).sum())


# Summarize the full graph
full_summary = summarize_graph(G_full)
print(f"Full TRRUST {SPECIES} graph summary:")
for key, value in full_summary.items():
    print(f"  {key}: {value}")

## 8. Analyze TF and target gene statistics

In [ ]:
# Analyze TF and target statistics
tf_counts = trrust_filtered["TF"].value_counts()
target_counts = trrust_filtered["Target"].value_counts()

print(f"Number of unique TFs: {len(tf_counts)}")
print(f"Number of unique targets: {len(target_counts)}")
print(f"Number of unique genes (TF or target): {len(set(trrust_filtered['TF']) | set(trrust_filtered['Target']))}")

print(f"\nTop 10 TFs by number of targets:")
print(tf_counts.head(10))

print(f"\nTop 10 most regulated targets:")
print(target_counts.head(10))

# Plot distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TF out-degree distribution
axes[0].hist(tf_counts.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of targets per TF')
axes[0].set_ylabel('Frequency')
axes[0].set_title('TF Out-degree Distribution')
axes[0].set_yscale('log')

# Target in-degree distribution
axes[1].hist(target_counts.values, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of TFs per target')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Target In-degree Distribution')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 9. Degree-aware subgraph sampling functions

In [ ]:
def induce_subgraph_by_nodes(G: nx.Graph, nodes: Iterable) -> nx.Graph:
    node_set = set(nodes)
    H = nx.Graph()
    H.add_nodes_from((n, G.nodes[n]) for n in node_set)
    H.add_edges_from((u, v, d) for u, v, d in G.edges(data=True) if u in node_set and v in node_set)
    return H


def weighted_choice_without_replacement(items, weights, k, rng):
    items = list(items)
    weights = np.asarray(weights, dtype=float)
    if len(items) == 0:
        return []
    if np.all(weights <= 0):
        weights = np.ones(len(items), dtype=float)
    weights = np.maximum(weights, 1e-12)
    probs = weights / weights.sum()
    k_eff = min(k, len(items))
    idx = rng.choice(len(items), size=k_eff, replace=False, p=probs)
    return [items[i] for i in idx]


def sample_anchor_node(G: nx.Graph, rng: np.random.Generator, power: float = 0.5):
    nodes = list(G.nodes())
    deg = np.array([G.degree(n) for n in nodes], dtype=float)
    weights = np.power(np.maximum(deg, 1.0), power)
    weights = weights / weights.sum()
    return nodes[int(rng.choice(len(nodes), p=weights))]


def sample_degree_targets(G: nx.Graph, sample_size: int, rng: np.random.Generator) -> np.ndarray:
    deg = graph_degree_array(G)
    if len(deg) == 0:
        return np.zeros(sample_size)
    return rng.choice(deg, size=sample_size, replace=True)


def connected_degree_aware_subgraph(
    G: nx.Graph,
    target_size: int,
    rng: np.random.Generator,
    restart_prob: float = 0.15,
    degree_weight_power: float = 0.5,
    frontier_width: int = 256,
    local_bfs_expansion: int = 8,
) -> nx.Graph:
    if target_size >= G.number_of_nodes():
        return G.copy()

    anchor = sample_anchor_node(G, rng, power=degree_weight_power)
    selected = [anchor]
    selected_set = {anchor}
    frontier = set(G.adj[anchor].keys())

    degree_targets = sample_degree_targets(G, target_size, rng)
    target_ptr = 0

    while len(selected) < target_size:
        if not frontier or rng.random() < restart_prob:
            seed_from_selected = selected[int(rng.integers(0, len(selected)))]
            local_frontier = deque([seed_from_selected])
            steps = 0
            while local_frontier and steps < local_bfs_expansion:
                u = local_frontier.popleft()
                nbrs = list(G.adj[u].keys())
                rng.shuffle(nbrs)
                for v in nbrs:
                    if v not in selected_set:
                        frontier.add(v)
                        local_frontier.append(v)
                steps += 1

        if not frontier:
            remaining = list(set(G.nodes()) - selected_set)
            if not remaining:
                break
            candidate = remaining[int(rng.integers(0, len(remaining)))]
            frontier.add(candidate)

        frontier_list = list(frontier)
        if len(frontier_list) > frontier_width:
            frontier_list = weighted_choice_without_replacement(
                frontier_list,
                [max(G.degree(n), 1) for n in frontier_list],
                frontier_width,
                rng,
            )

        target_degree = degree_targets[min(target_ptr, len(degree_targets) - 1)]
        scores = []
        for node in frontier_list:
            deg = G.degree(node)
            internal_links = sum((nbr in selected_set) for nbr in G.adj[node].keys())
            degree_match = 1.0 / (1.0 + abs(deg - target_degree))
            score = 2.0 * internal_links + degree_match + 0.25 * np.log1p(deg)
            scores.append(score)

        scores = np.asarray(scores, dtype=float)
        if np.all(scores <= 0):
            scores = np.ones_like(scores)
        probs = scores / scores.sum()
        chosen_idx = int(rng.choice(len(frontier_list), p=probs))
        chosen = frontier_list[chosen_idx]

        selected.append(chosen)
        selected_set.add(chosen)
        frontier.discard(chosen)
        frontier.update(v for v in G.adj[chosen].keys() if v not in selected_set)
        target_ptr += 1

    H = induce_subgraph_by_nodes(G, selected_set)
    H = keep_largest_connected_component(H)

    # If taking LCC dropped too many nodes, greedily refill through the boundary
    while H.number_of_nodes() < target_size and H.number_of_nodes() < G.number_of_nodes():
        current_nodes = set(H.nodes())
        boundary = set()
        for u in current_nodes:
            boundary.update(v for v in G.adj[u].keys() if v not in current_nodes)
        if not boundary:
            break
        boundary = list(boundary)
        rng.shuffle(boundary)
        needed = min(target_size - H.number_of_nodes(), len(boundary))
        current_nodes.update(boundary[:needed])
        H = keep_largest_connected_component(induce_subgraph_by_nodes(G, current_nodes))

    return H


def repeated_connected_subgraph_sampling(
    G: nx.Graph,
    target_sizes: Sequence[int],
    repeats: int,
    base_seed: int = 42,
    restart_prob: float = 0.15,
    degree_weight_power: float = 0.5,
    frontier_width: int = 256,
    local_bfs_expansion: int = 8,
) -> List[Dict]:
    results = []
    for target_size in target_sizes:
        effective_target = min(target_size, G.number_of_nodes())
        for repeat_idx in range(repeats):
            seed = base_seed + 1000 * repeat_idx + int(target_size)
            rng = np.random.default_rng(seed)
            H = connected_degree_aware_subgraph(
                G,
                target_size=effective_target,
                rng=rng,
                restart_prob=restart_prob,
                degree_weight_power=degree_weight_power,
                frontier_width=frontier_width,
                local_bfs_expansion=local_bfs_expansion,
            )
            dist = degree_histogram_distance(G, H)
            results.append({
                "target_size": int(target_size),
                "effective_target_size": int(effective_target),
                "repeat_idx": int(repeat_idx),
                "seed": int(seed),
                "graph": H,
                "summary": summarize_graph(H),
                "degree_hist_l1": float(dist),
            })
    return results

## 10. Run repeated sampling

In [ ]:
samples = repeated_connected_subgraph_sampling(
    G_full,
    target_sizes=TARGET_SIZES,
    repeats=REPEATS,
    base_seed=BASE_SEED,
    restart_prob=RESTART_PROB,
    degree_weight_power=DEGREE_WEIGHT_POWER,
    frontier_width=FRONTIER_WIDTH,
    local_bfs_expansion=LOCAL_BFS_EXPANSION,
)

print(f"Generated {len(samples)} sampled subgraphs")
print("\nFirst 5 samples:")
for s in samples[:5]:
    print(f"  Target: {s['target_size']}, Repeat: {s['repeat_idx']}, "
          f"Nodes: {s['summary']['num_nodes']}, Edges: {s['summary']['num_edges']}")

## 11. Review sample quality

In [ ]:
quality_rows = []
for s in samples:
    row = {
        "target_size": s["target_size"],
        "effective_target_size": s["effective_target_size"],
        "repeat_idx": s["repeat_idx"],
        "seed": s["seed"],
        "degree_hist_l1": s["degree_hist_l1"],
    }
    row.update(s["summary"])
    quality_rows.append(row)

quality_df = pd.DataFrame(quality_rows).sort_values(["target_size", "repeat_idx"])
quality_df

In [ ]:
# Summary statistics by target size
quality_df.groupby("target_size")["degree_hist_l1"].agg(["mean", "std", "min", "max"]).reset_index()

In [ ]:
# Best sample per target size (lowest degree histogram distance)
best_per_size = (
    quality_df.sort_values("degree_hist_l1")
    .groupby("target_size", as_index=False)
    .first()
)
best_per_size

In [ ]:
# Plot degree distributions
graphs_to_plot = {"full_graph": G_full}
for _, row in best_per_size.iterrows():
    match = [
        s for s in samples
        if s["target_size"] == row["target_size"] and s["repeat_idx"] == row["repeat_idx"]
    ][0]
    graphs_to_plot[f"sample_{int(row['target_size'])}_rep{int(row['repeat_idx'])}"] = match["graph"]

plot_degree_ccdf(graphs_to_plot, title=f"TRRUST {SPECIES}: Full graph vs best sampled subgraphs")

## 12. Save sampled subgraphs for QuVINE

In [ ]:
def save_graph_edgelist_csv(G: nx.Graph, path: Path):
    df = pd.DataFrame(list(G.edges()), columns=["node1", "node2"])
    df.to_csv(path, index=False)


def save_sample_bundle(sample: Dict, output_dir: Path, dataset_name: str, species: str):
    graph = sample["graph"]
    target_size = sample["target_size"]
    repeat_idx = sample["repeat_idx"]
    stem = f"{dataset_name}_{species}_n{target_size}_rep{repeat_idx}"

    csv_path = output_dir / f"{stem}.csv"
    json_path = output_dir / f"{stem}.json"

    save_graph_edgelist_csv(graph, csv_path)
    metadata = {
        "dataset_name": dataset_name,
        "species": species,
        "data_source": "TRRUST v2",
        "drop_unknown_effects": DROP_UNKNOWN,
        "target_size": int(sample["target_size"]),
        "effective_target_size": int(sample["effective_target_size"]),
        "repeat_idx": int(sample["repeat_idx"]),
        "seed": int(sample["seed"]),
        "degree_hist_l1": float(sample["degree_hist_l1"]),
        "summary": sample["summary"],
        "sampling_parameters": {
            "restart_prob": RESTART_PROB,
            "degree_weight_power": DEGREE_WEIGHT_POWER,
            "frontier_width": FRONTIER_WIDTH,
            "local_bfs_expansion": LOCAL_BFS_EXPANSION,
            "base_seed": BASE_SEED,
        },
    }
    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)
    return csv_path, json_path


# Save all samples
saved_paths = []
for sample in tqdm(samples, desc="Saving samples"):
    saved_paths.append(save_sample_bundle(sample, OUTPUT_DIR, "trrust", SPECIES))

print(f"\nSaved {len(saved_paths)} sample bundles to {OUTPUT_DIR}")
print("\nFirst 5 saved files:")
for csv_path, json_path in saved_paths[:5]:
    print(f"  {csv_path.name}")

## 13. Optional: Save only best samples per target size

In [ ]:
# If you want to save only the best sample per target size
best_samples = []
for target_size in sorted(set(s["target_size"] for s in samples)):
    subset = [s for s in samples if s["target_size"] == target_size]
    best = min(subset, key=lambda x: x["degree_hist_l1"])
    best_samples.append(best)

print("Best samples per target size:")
for s in best_samples:
    print(f"  Target: {s['target_size']}, Repeat: {s['repeat_idx']}, "
          f"Degree L1: {s['degree_hist_l1']:.4f}")

## 14. Summary and next steps

### What we did:
1. Downloaded TRRUST v2 TF-target interactions (human or mouse)
2. Analyzed signed regulation (Activation, Repression, Unknown)
3. Optionally filtered out "Unknown" effects
4. Built an undirected graph from the interactions
5. Sampled multiple connected subgraphs with degree-aware sampling
6. Saved all samples as CSV edgelists + JSON metadata

### Dataset characteristics:
- **Species**: Human or Mouse
- **Regulation types**: Activation, Repression, (optionally Unknown)
- **Literature-curated**: Each interaction has PubMed references
- **Directed nature**: TF → Target (converted to undirected for sampling)

### Next steps for QuVINE:
1. Point QuVINE graph-loading config to one of the saved CSV files
2. Run `prepare_graph(...)` if you want further sparsification/subsampling
3. Evaluate embeddings across multiple repeats of the sampled subgraphs
4. Compare with DoRothEA and other regulatory network benchmarks

### Example usage:
```python
import pandas as pd
import networkx as nx

# Load a sample
df = pd.read_csv("../data/processed_data/trrust_human_graph_samples/trrust_human_n5000_rep0.csv")
G = nx.from_pandas_edgelist(df, source="node1", target="node2")
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
```